In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

In [2]:
DATA_PATH = 'archive/dataset/dataset/Annual_P_L_1_final.csv'
OUTPUT_PATH = 'output/opm_classification/'

In [3]:
df = pd.read_csv(DATA_PATH)
print(f"Loaded: {df.shape[0]} rows × {df.shape[1]} columns")

Loaded: 4668 rows × 58 columns


In [4]:
print(f"\nOPM Statistics:")
print(df['OPM'].describe())


OPM Statistics:
count      4367.000000
mean        -89.535244
std        2264.934140
min     -102800.000000
25%           2.780000
50%           9.910000
75%          19.280000
max        3094.120000
Name: OPM, dtype: float64


In [5]:
print(f"\nOPM Distribution:")
print(f"   Missing values: {df['OPM'].isna().sum()} ({df['OPM'].isna().sum()/len(df)*100:.1f}%)")
print(f"   Min: {df['OPM'].min():.2f}%")
print(f"   Max: {df['OPM'].max():.2f}%")
print(f"   Mean: {df['OPM'].mean():.2f}%")
print(f"   Median: {df['OPM'].median():.2f}%")


OPM Distribution:
   Missing values: 301 (6.4%)
   Min: -102800.00%
   Max: 3094.12%
   Mean: -89.54%
   Median: 9.91%


In [6]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
df['OPM'].hist(bins=100, edgecolor='black')
plt.xlabel('OPM (%)')
plt.ylabel('Frequency')
plt.title('OPM Distribution (Raw)')
plt.axvline(x=0, color='red', linestyle='--', label='Zero line')
plt.axvline(x=10, color='orange', linestyle='--', label='10%')
plt.axvline(x=20, color='green', linestyle='--', label='20%')
plt.legend()

plt.subplot(1, 2, 2)
df_opm_clean = df[(df['OPM'] >= -50) & (df['OPM'] <= 50)]
df_opm_clean['OPM'].hist(bins=50, edgecolor='black')
plt.xlabel('OPM (%)')
plt.ylabel('Frequency')
plt.title('OPM Distribution (Filtered -50% to 50%)')
plt.axvline(x=0, color='red', linestyle='--', label='Zero line')
plt.axvline(x=10, color='orange', linestyle='--', label='10%')
plt.axvline(x=20, color='green', linestyle='--', label='20%')
plt.legend()

plt.tight_layout()
os.makedirs(OUTPUT_PATH, exist_ok=True)
plt.savefig(f'{OUTPUT_PATH}01_opm_distribution_raw.png', dpi=300, bbox_inches='tight')
print(f"\n   Saved: {OUTPUT_PATH}01_opm_distribution_raw.png")
plt.close()


   Saved: output/opm_classification/01_opm_distribution_raw.png


In [7]:
print("\n🧹 STEP 1: Remove obvious data errors...")

df_filtered = df[
    (df['OPM'] > -100) &   # No company loses 100x more than it earns
    (df['OPM'] < 200)      # No company has 200% OPM
].copy()

n_removed_step1 = len(df) - len(df_filtered)
pct_removed_step1 = n_removed_step1 / len(df) * 100

print(f"   Removed {n_removed_step1} obvious errors ({pct_removed_step1:.2f}%)")
print(f"   Remaining: {len(df_filtered)} samples")


🧹 STEP 1: Remove obvious data errors...
   Removed 501 obvious errors (10.73%)
   Remaining: 4167 samples


In [8]:
print("\nSTEP 2: Winsorization...")
lower_percentile = 1
upper_percentile = 99

opm_lower = df_filtered['OPM'].quantile(lower_percentile/100)
opm_upper = df_filtered['OPM'].quantile(upper_percentile/100)

print(f"   {lower_percentile}th percentile: {opm_lower:.2f}%")
print(f"   {upper_percentile}th percentile: {opm_upper:.2f}%")

n_lower = (df_filtered['OPM'] < opm_lower).sum()
n_upper = (df_filtered['OPM'] > opm_upper).sum()
print(f"   Values to cap (lower): {n_lower}")
print(f"   Values to cap (upper): {n_upper}")

df_filtered['OPM_clean'] = df_filtered['OPM'].clip(opm_lower, opm_upper)


STEP 2: Winsorization...
   1th percentile: -61.06%
   99th percentile: 92.62%
   Values to cap (lower): 42
   Values to cap (upper): 42


In [9]:
print("\nSTEP 3: Apply business rules...")
original_lower = df_filtered['OPM_clean'].min()
original_upper = df_filtered['OPM_clean'].max()

BUSINESS_MIN = -100
BUSINESS_MAX = 120

if opm_lower < BUSINESS_MIN:
    print(f"Lower bound ({opm_lower:.2f}%) too extreme, capping at {BUSINESS_MIN}%")
    df_filtered['OPM_clean'] = df_filtered['OPM_clean'].clip(lower=BUSINESS_MIN)

if opm_upper > BUSINESS_MAX:
    print(f"Upper bound ({opm_upper:.2f}%) too extreme, capping at {BUSINESS_MAX}%")
    df_filtered['OPM_clean'] = df_filtered['OPM_clean'].clip(upper=BUSINESS_MAX)

# Final statistics
print("\nCLEANING SUMMARY:")
print(f"   Original samples:     {len(df)}")
print(f"   After cleaning:       {len(df_filtered)}")
print(f"   Removed total:        {len(df) - len(df_filtered)} ({(len(df) - len(df_filtered))/len(df)*100:.2f}%)")
print(f"\n   Original OPM range:   [{df['OPM'].min():.2f}%, {df['OPM'].max():.2f}%]")
print(f"   Cleaned OPM range:    [{df_filtered['OPM_clean'].min():.2f}%, {df_filtered['OPM_clean'].max():.2f}%]")
print(f"\n   Original mean:        {df['OPM'].mean():.2f}%")
print(f"   Cleaned mean:         {df_filtered['OPM_clean'].mean():.2f}%")
print(f"   Original median:      {df['OPM'].median():.2f}%")
print(f"   Cleaned median:       {df_filtered['OPM_clean'].median():.2f}%")
print(f"\n   Original std:         {df['OPM'].std():.2f}%")
print(f"   Cleaned std:          {df_filtered['OPM_clean'].std():.2f}%")


STEP 3: Apply business rules...

CLEANING SUMMARY:
   Original samples:     4668
   After cleaning:       4167
   Removed total:        501 (10.73%)

   Original OPM range:   [-102800.00%, 3094.12%]
   Cleaned OPM range:    [-61.06%, 92.62%]

   Original mean:        -89.54%
   Cleaned mean:         13.80%
   Original median:      9.91%
   Cleaned median:       10.54%

   Original std:         2264.93%
   Cleaned std:          24.04%


In [10]:
df = df_filtered
df_clean = df

In [15]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['OPM'].clip(-50, 100), bins=50, edgecolor='black', alpha=0.7)
axes[0].axvline(x=0, color='red', linestyle='--', linewidth=2, label='Zero')
axes[0].axvline(x=10, color='orange', linestyle='--', linewidth=2, label='10%')
axes[0].axvline(x=20, color='green', linestyle='--', linewidth=2, label='20%')
axes[0].set_xlabel('OPM (%)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('OPM Distribution - Before Cleaning\n(displayed -50% to 100%)')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# After
axes[1].hist(df['OPM_clean'], bins=50, edgecolor='black', alpha=0.7, color='green')
axes[1].axvline(x=0, color='red', linestyle='--', linewidth=2, label='Zero')
axes[1].axvline(x=10, color='orange', linestyle='--', linewidth=2, label='10%')
axes[1].axvline(x=20, color='green', linestyle='--', linewidth=2, label='20%')
axes[1].set_xlabel('OPM (%)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('OPM Distribution - After Cleaning')
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
os.makedirs(OUTPUT_PATH, exist_ok=True)
plt.savefig(f'{OUTPUT_PATH}01_opm_distribution_before_after.png', dpi=300, bbox_inches='tight')
print(f"\n   Saved: {OUTPUT_PATH}01_opm_distribution_before_after.png")
plt.close()


   Saved: output/opm_classification/01_opm_distribution_before_after.png


In [17]:
created_features = []

# ========== 1. GROWTH METRICS ==========
print("\n1. Growth metrics...")

# Sales growth
if 'Sales' in df.columns and 'Sales last year' in df.columns:
    df['sales_growth'] = (
        (df['Sales'] - df['Sales last year']) / 
        df['Sales last year'].replace(0, np.nan) * 100
    )
    created_features.append('sales_growth')
    print("   sales_growth")

# Profit growth
if 'Profit after tax' in df.columns and 'Profit after tax last year' in df.columns:
    df['profit_growth'] = (
        (df['Profit after tax'] - df['Profit after tax last year']) / 
        df['Profit after tax last year'].replace(0, np.nan) * 100
    )
    created_features.append('profit_growth')
    print("   profit_growth")

# OPM change
if 'OPM_clean' in df.columns and 'OPM last year' in df.columns:
    df['opm_change'] = df['OPM_clean'] - df['OPM last year']
    created_features.append('opm_change')
    print("   opm_change")

# ========== 2. COST STRUCTURE RATIOS ==========
print("\n2️. Cost structure ratios...")

# Material cost ratio
if 'Material cost last year' in df.columns and 'Sales last year' in df.columns:
    df['material_cost_ratio'] = (
        df['Material cost last year'] / 
        df['Sales last year'].replace(0, np.nan) * 100
    )
    created_features.append('material_cost_ratio')
    print("   material_cost_ratio")

# Employee cost ratio
if 'Employee cost last year' in df.columns and 'Sales last year' in df.columns:
    df['employee_cost_ratio'] = (
        df['Employee cost last year'] / 
        df['Sales last year'].replace(0, np.nan) * 100
    )
    created_features.append('employee_cost_ratio')
    print("   employee_cost_ratio")

# Total cost ratio
if 'material_cost_ratio' in df.columns and 'employee_cost_ratio' in df.columns:
    df['total_cost_ratio'] = (
        df['material_cost_ratio'] + df['employee_cost_ratio']
    )
    created_features.append('total_cost_ratio')
    print("   total_cost_ratio")

# Depreciation ratio
if 'Depreciation' in df.columns and 'Sales' in df.columns:
    df['depreciation_ratio'] = (
        df['Depreciation'] / 
        df['Sales'].replace(0, np.nan) * 100
    )
    created_features.append('depreciation_ratio')
    print("   depreciation_ratio")

# ========== 3. EFFICIENCY METRICS ==========
print("\n3️. Efficiency metrics...")

# Asset turnover
if 'Sales' in df.columns and 'Market Capitalization' in df.columns:
    df['asset_turnover'] = (
        df['Sales'] / 
        df['Market Capitalization'].replace(0, np.nan)
    )
    created_features.append('asset_turnover')
    print("   asset_turnover")

# ========== 4. SIZE INDICATORS (LOG SCALE) ==========
print("\n4️. Size indicators (log scale)...")

# Log sales
if 'Sales' in df.columns:
    df['log_sales'] = np.log1p(df['Sales'].clip(lower=0))
    created_features.append('log_sales')
    print("   log_sales")

# Log market cap
if 'Market Capitalization' in df.columns:
    df['log_market_cap'] = np.log1p(
        df['Market Capitalization'].clip(lower=0)
    )
    created_features.append('log_market_cap')
    print("   log_market_cap")

# ========== 5. HISTORICAL PERFORMANCE INDICATORS ==========
print("\n5️. Historical performance indicators...")

# Profitable last year
if 'Profit after tax last year' in df.columns:
    df['profitable_last_year'] = (
        df['Profit after tax last year'] > 0
    ).astype(int)
    created_features.append('profitable_last_year')
    print("   profitable_last_year")

# Positive OPM last year
if 'OPM last year' in df.columns:
    df['positive_opm_last_year'] = (
        df['OPM last year'] > 0
    ).astype(int)
    created_features.append('positive_opm_last_year')
    print("   positive_opm_last_year")

# Improving OPM
if 'opm_change' in df.columns:
    df['improving_opm'] = (
        df['opm_change'] > 0
    ).astype(int)
    created_features.append('improving_opm')
    print("   improving_opm")

# ========== SUMMARY ==========
print(f"\nFeature engineering summary:")
print(f"   Created features: {len(created_features)}")
print(f"   Expected: 12-13 features")

if len(created_features) >= 12:
    print(f"   All features created successfully!")
else:
    print(f"   Some features missing - check column availability")

# Show statistics for created features
print(f"\nCreated features statistics:")
for feat in created_features[:5]:
    if feat in df.columns:
        print(f"   {feat:25} | Mean: {df[feat].mean():8.2f} | Std: {df[feat].std():8.2f}")


1. Growth metrics...
   sales_growth
   profit_growth
   opm_change

2️. Cost structure ratios...
   material_cost_ratio
   employee_cost_ratio
   total_cost_ratio
   depreciation_ratio

3️. Efficiency metrics...
   asset_turnover

4️. Size indicators (log scale)...
   log_sales
   log_market_cap

5️. Historical performance indicators...
   profitable_last_year
   positive_opm_last_year
   improving_opm

Feature engineering summary:
   Created features: 13
   Expected: 12-13 features
   All features created successfully!

Created features statistics:
   sales_growth              | Mean:     0.45 | Std:    23.44
   profit_growth             | Mean:    -2.96 | Std:   196.64
   opm_change                | Mean:     0.12 | Std:     6.44
   material_cost_ratio       | Mean:   331.54 | Std:  3422.36
   employee_cost_ratio       | Mean:    13.24 | Std:    15.34


In [13]:
print("\n" + "="*80)
print("FEATURE SELECTION (Anti-Leakage)")
print("="*80)

# Core features for OPM prediction (WITHOUT data leakage)
feature_cols = [
    # ========== COST STRUCTURE ==========
    'Material cost last year',
    'Employee cost last year',
    'material_cost_ratio',      # Created in feature engineering
    'employee_cost_ratio',      # Created in feature engineering
    'total_cost_ratio',         # Created in feature engineering
    'Depreciation',
    'depreciation_ratio',       # Created in feature engineering
    
    # ========== HISTORICAL PERFORMANCE ==========
    'OPM last year',            # ⭐ Most important!
    'Profit after tax last year',
    'Sales last year',
    'positive_opm_last_year',   # Created in feature engineering
    'profitable_last_year',     # Created in feature engineering
    
    # ========== COMPANY SIZE ==========
    'log_sales',                # Created in feature engineering
    'log_market_cap',           # Created in feature engineering
    
    # ========== GROWTH & MOMENTUM ==========
    'sales_growth',             # Created in feature engineering
    'opm_change',               # Created in feature engineering
    'improving_opm',            # Created in feature engineering
    
    # ========== EFFICIENCY ==========
    'Return on capital employed',
    'asset_turnover',           # Created in feature engineering
    
    # ========== FINANCIAL STRUCTURE ==========
    'Interest',
    
    # ========== INDUSTRY ==========
    'Industry'                  # Will be encoded later
]

# Check which features are available
available_features = [f for f in feature_cols if f in df_clean.columns]
missing_features = [f for f in feature_cols if f not in df_clean.columns]

print(f"\n📋 Feature inventory:")
print(f"   Requested: {len(feature_cols)} features")
print(f"   Available: {len(available_features)} features")

if missing_features:
    print(f"\n   ⚠️  Missing features: {len(missing_features)}")
    for feat in missing_features:
        print(f"      - {feat}")

print(f"\n📊 Available features:")
for i, feat in enumerate(available_features, 1):
    dtype = df_clean[feat].dtype
    missing = df_clean[feat].isna().sum()
    missing_pct = missing / len(df_clean) * 100
    
    # Categorize
    if 'cost' in feat.lower() or 'ratio' in feat.lower():
        category = 'COST'
    elif 'last year' in feat.lower() or 'positive' in feat.lower() or 'profitable' in feat.lower():
        category = 'HIST'
    elif 'log' in feat.lower():
        category = 'SIZE'
    elif 'growth' in feat.lower() or 'change' in feat.lower() or 'improving' in feat.lower():
        category = 'GROW'
    elif feat == 'Industry':
        category = 'CATG'
    else:
        category = 'OTHR'
    
    print(f"   {i:2}. [{category}] {feat:35} | Missing: {missing:4} ({missing_pct:4.1f}%)")

# Count by category
categories = {
    'COST': 'Cost Structure',
    'HIST': 'Historical Performance', 
    'SIZE': 'Company Size',
    'GROW': 'Growth & Momentum',
    'CATG': 'Categorical',
    'OTHR': 'Other'
}

print(f"\n📦 Features by category:")
for cat_code, cat_name in categories.items():
    count = sum(1 for feat in available_features 
                if (cat_code in feat.upper() if cat_code == 'CATG' else
                    (cat_code == 'COST' and ('cost' in feat.lower() or 'ratio' in feat.lower())) or
                    (cat_code == 'HIST' and ('last year' in feat.lower() or 'positive' in feat.lower() or 'profitable' in feat.lower())) or
                    (cat_code == 'SIZE' and 'log' in feat.lower()) or
                    (cat_code == 'GROW' and ('growth' in feat.lower() or 'change' in feat.lower() or 'improving' in feat.lower())) or
                    (cat_code == 'OTHR' and cat_code not in ['COST', 'HIST', 'SIZE', 'GROW', 'CATG'])))
    if count > 0:
        print(f"   {cat_name:25} → {count:2} features")

print(f"\n✅ Total clean features: {len(available_features)}")
print(f"   No data leakage - all features are predictive, not derived from OPM")


FEATURE SELECTION (Anti-Leakage)

📋 Feature inventory:
   Requested: 21 features
   Available: 21 features

📊 Available features:
    1. [COST] Material cost last year             | Missing:    0 ( 0.0%)
    2. [COST] Employee cost last year             | Missing:    0 ( 0.0%)
    3. [COST] material_cost_ratio                 | Missing:    0 ( 0.0%)
    4. [COST] employee_cost_ratio                 | Missing:    0 ( 0.0%)
    5. [COST] total_cost_ratio                    | Missing:    0 ( 0.0%)
    6. [OTHR] Depreciation                        | Missing:    0 ( 0.0%)
    7. [COST] depreciation_ratio                  | Missing:    0 ( 0.0%)
    8. [HIST] OPM last year                       | Missing:    0 ( 0.0%)
    9. [HIST] Profit after tax last year          | Missing:    0 ( 0.0%)
   10. [HIST] Sales last year                     | Missing:    0 ( 0.0%)
   11. [HIST] positive_opm_last_year              | Missing:    0 ( 0.0%)
   12. [HIST] profitable_last_year                | Mis